# Day 3 — LangGraph: Stateful Agent Workflows

---

Your hand-rolled ReAct loop works — for one linear flow. Real agents need:

- **Branching** — "if the search returned nothing, try a different query"
- **Loops with clear exit conditions**
- **Resumability** — pause for human input, then continue
- **Debuggability** — see exactly what each step did

That's what **LangGraph** gives you. It's a small library from the LangChain team that models an agent as a **graph of nodes** with typed state flowing between them.

Not the only option — LlamaIndex Workflows and Pydantic AI are close alternatives. LangGraph is the most widely used in 2026 job listings, so it's what we'll teach.


## 1. The mental model — graph, state, nodes, edges

- **State** — a Python dict (usually a `TypedDict`) that flows through the graph
- **Node** — a function that takes state and returns updates to state
- **Edge** — connects one node to the next; can be conditional
- **Graph** — the whole thing, compiled into an executor

```
       ┌────────┐
       │  plan  │
       └────┬───┘
            │
            ▼
       ┌────────┐   yes    ┌────────┐
       │ search │ ───────► │  done  │
       └────┬───┘          └────────┘
            │ no
            ▼
       ┌────────┐
       │ retry  │
       └────────┘
```

Each box is a node. Arrows are edges. That's the whole abstraction.


## 2. Setup


In [ ]:
!pip install langgraph together python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from together import Together
llm = Together()


## 3. Define the state


In [ ]:
from typing import TypedDict, Annotated
import operator

class AgentState(TypedDict):
    question: str
    plan: str
    steps_taken: Annotated[list[str], operator.add]  # accumulates across nodes
    answer: str


The `Annotated[list, operator.add]` is a LangGraph pattern that says: *"when nodes update this field, append instead of overwrite."* Perfect for logs / step history.


## 4. Define the nodes

Each node is a plain Python function that takes state and returns a dict of updates.


In [ ]:
def plan_node(state: AgentState) -> dict:
    prompt = f"Question: {state['question']}\nWrite a 1-2 sentence plan for how to answer this."
    resp = llm.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    plan = resp.choices[0].message.content.strip()
    return {"plan": plan, "steps_taken": ["planned"]}


def act_node(state: AgentState) -> dict:
    prompt = (f"Question: {state['question']}\nPlan: {state['plan']}\n"
              "Now write a concise final answer.")
    resp = llm.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return {"answer": resp.choices[0].message.content.strip(),
            "steps_taken": ["acted"]}


def review_node(state: AgentState) -> dict:
    prompt = (f"Question: {state['question']}\nAnswer: {state['answer']}\n"
              "Rewrite the answer to be crisp and under 50 words.")
    resp = llm.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
    )
    return {"answer": resp.choices[0].message.content.strip(),
            "steps_taken": ["reviewed"]}


## 5. Build the graph


In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)
builder.add_node("plan",   plan_node)
builder.add_node("act",    act_node)
builder.add_node("review", review_node)

builder.add_edge(START,    "plan")
builder.add_edge("plan",   "act")
builder.add_edge("act",    "review")
builder.add_edge("review", END)

graph = builder.compile()


## 6. Run it


In [ ]:
final = graph.invoke({"question": "Explain HTTPS in one paragraph."})

print("Steps:", final["steps_taken"])
print("\nPlan:", final["plan"])
print("\nAnswer:", final["answer"])


**Notice:** you didn't write a loop. You wired up three nodes, LangGraph runs them in order, and the state accumulates.

For a linear 3-step flow this feels like overkill (and it is). The win comes with branches and loops.


## 7. Conditional edges (the real reason to use a framework)

Say we want to re-plan if the answer looks weak. Add a **conditional edge**:


In [ ]:
def is_answer_ok(state: AgentState) -> str:
    # Toy check: reject one-line answers
    return "end" if len(state["answer"]) > 40 else "retry"

builder2 = StateGraph(AgentState)
builder2.add_node("plan",   plan_node)
builder2.add_node("act",    act_node)
builder2.add_edge(START,   "plan")
builder2.add_edge("plan",  "act")
builder2.add_conditional_edges("act", is_answer_ok, {"end": END, "retry": "plan"})
graph2 = builder2.compile()

# Try it
r = graph2.invoke({"question": "What is 2+2?"})
print("Steps:", r["steps_taken"])
print("Answer:", r["answer"])


If the answer is short, the graph loops back to `plan`. LangGraph handles the branching for you — you'd hand-write this in the raw loop.


## 8. Tools inside a graph (the standard agent pattern)

LangGraph ships a `create_react_agent` helper that wraps the whole "loop until no more tool calls" agent into one node. In practice you use it whenever you don't need custom branching:

```python
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

agent = create_react_agent(some_llm, [add])
result = agent.invoke({"messages": [("user", "What is 3+5?")]})
```

We won't dive deep here — the point is: once you know graphs + nodes + tools, you have every pattern in the framework's playbook.


## 9. Multi-agent — AutoGen, CrewAI (one paragraph)

**AutoGen** (Microsoft) and **CrewAI** are libraries for orchestrating **multiple agents talking to each other** — e.g. a "researcher" agent, a "writer" agent, a "critic" agent. They're powerful and easy to over-use.

**Fresher advice:** don't reach for multi-agent unless you have a *concrete* reason (usually: distinct role prompts + tool sets). Most "multi-agent" problems are better solved as one agent with a LangGraph plan. Know the names, save the deep dive for a real project need.


## Recap

- **LangGraph** models an agent as **state + nodes + edges**.
- Nodes are plain Python functions. State is a `TypedDict`.
- **Conditional edges** are the killer feature — branches and loops with clear exit conditions.
- `create_react_agent` is a batteries-included shortcut for the "loop while there are tool calls" pattern.
- **AutoGen / CrewAI** = multi-agent orchestration. Know the names, use sparingly.
- **Next class:** memory — short-term (this conversation) vs long-term (past sessions in a vector store).
